# 🔍 Notebook 2 — TF-IDF Baseline
**Project:** Multilingual Fake News Detection  
**Author:** Asliddin | Presidential School, Namangan

---
Classical ML baseline: **TF-IDF features + Logistic Regression**.  
Fast, interpretable, and sets our performance floor before transformers.

**Expected results:** ~78% accuracy, Macro F1 ~0.76 (English only)

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report

from model import BaselineModel
from evaluate import compute_metrics, plot_confusion_matrix

print('Libraries loaded ✓')

## 1. Load Data

In [ ]:
LABEL2ID = {'real': 0, 'fake': 1, 'satire': 2}

train_df = pd.read_csv('../data/processed/train.csv')
val_df   = pd.read_csv('../data/processed/val.csv')
test_df  = pd.read_csv('../data/processed/test.csv')

# Baseline is English-only (TF-IDF doesn't generalize well cross-lingually)
train_en = train_df[train_df['lang'] == 'en']
val_en   = val_df[val_df['lang'] == 'en']
test_en  = test_df[test_df['lang'] == 'en']

print(f'English train: {len(train_en):,} | val: {len(val_en):,} | test: {len(test_en):,}')

X_train = train_en['text'].tolist()
y_train = train_en['label'].map(LABEL2ID).tolist()
X_val   = val_en['text'].tolist()
y_val   = val_en['label'].map(LABEL2ID).tolist()
X_test  = test_en['text'].tolist()
y_test  = test_en['label'].map(LABEL2ID).tolist()

## 2. Train Baseline

In [ ]:
baseline = BaselineModel(max_features=50000, ngram_range=(1, 2))
baseline.fit(X_train, y_train)

val_preds = baseline.predict(X_val)
val_metrics = compute_metrics(y_val, val_preds)

print(f'\nValidation Results:')
for k, v in val_metrics.items():
    print(f'  {k.capitalize():>10}: {v:.4f}')

baseline.save('../models/baseline.pkl')

## 3. Test Set Evaluation

In [ ]:
test_preds = baseline.predict(X_test)
test_metrics = compute_metrics(y_test, test_preds)

print('Test Results:')
for k, v in test_metrics.items():
    print(f'  {k.capitalize():>10}: {v:.4f}')

print('\nDetailed Report:')
print(classification_report(y_test, test_preds, target_names=['Real','Fake','Satire']))

plot_confusion_matrix(y_test, test_preds,
                      save_path='../results/baseline_confusion_matrix.png')

## 4. Top TF-IDF Features per Class

In [ ]:
# Inspect which words are most predictive per class
tfidf    = baseline.pipeline.named_steps['tfidf']
clf      = baseline.pipeline.named_steps['clf']
features = tfidf.get_feature_names_out()

class_names = ['Real', 'Fake', 'Satire']
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Top 15 Most Predictive TF-IDF Features per Class',
             fontsize=13, fontweight='bold')

colors = ['#2d6a4f', '#d62828', '#f4a261']

for i, (ax, cls_name, color) in enumerate(zip(axes, class_names, colors)):
    coef      = clf.coef_[i]
    top_idx   = coef.argsort()[-15:][::-1]
    top_words = [features[j] for j in top_idx]
    top_coefs = coef[top_idx]

    ax.barh(range(15), top_coefs, color=color, alpha=0.85, edgecolor='white')
    ax.set_yticks(range(15))
    ax.set_yticklabels(top_words, fontsize=9)
    ax.set_title(f'{cls_name} indicators', fontweight='bold', color=color)
    ax.set_xlabel('Coefficient')
    ax.grid(axis='x', alpha=0.3)
    ax.invert_yaxis()

plt.tight_layout()
plt.savefig('../results/tfidf_features.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n→ Next: Notebook 03 — XLM-RoBERTa fine-tuning for multilingual classification')